# Complete Active Learning Cycle

This notebook demonstrates the complete workflow from GPR predictions to DFT calculations and back to training.

## Workflow Overview

```
Cycle N:
1. Train GPR on existing DFT data → Get predictions
2. Select top candidates → batch{N}_metal.csv
3. Run DFT calculations → DFT_O_batch{N}.csv, DFT_OH_batch{N}.csv
4. Combine with existing data → DFT_O_all.csv, DFT_OH_all.csv
5. Repeat for Cycle N+1
```

## Part 1: GPR Training and Batch Selection

In [ ]:
from active_learning import (
    DataspaceGenerator,
    DFTCompatibleGenerator,
    ActiveLearner,
    ElementCounter,
    ActivityCalculator
)
import numpy as np
import pandas as pd

# Set batch number
BATCH = 15

print(f"Active Learning Cycle {BATCH}")

In [ ]:
# Step 1: Generate dataspace (only needed once)
gen = DataspaceGenerator(n_metals=3, zone_sizes=(3, 6, 3, 3, 3))
gen.generate('GPRdataspace.csv')

# Step 2: Generate DFT-compatible subset (only needed once)
dft_gen = DFTCompatibleGenerator()
dft_gen.generate('possibleFp.csv', 'index_metal.csv')

In [ ]:
# Step 3: Train GPR models on all existing data
learner = ActiveLearner(n_features=15)

# Load all DFT data collected so far (batches 1-14)
learner.load_data(
    o_file='DFT_O_all.csv',    # All O data from batches 1-14
    oh_file='DFT_OH_all.csv'   # All OH data from batches 1-14
)

print(f"Training data: {learner.n_train_o} O samples, {learner.n_train_oh} OH samples")

# Train models
learner.train()

print(f"\nModel Performance:")
print(f"  O model MAE: {learner.mae_o:.4f} eV")
print(f"  OH model MAE: {learner.mae_oh:.4f} eV")

In [ ]:
# Step 4: Generate batch 15 suggestions
suggestions = learner.suggest_next_batch(
    batch_number=BATCH,
    n_suggestions=30,
    dataspace_file='GPRdataspace.csv',
    possible_file='possibleFp.csv',
    index_file='index_metal.csv'
)

print(f"\n Generated {len(suggestions)} suggestions for batch {BATCH+1}")
print(f"  Saved to: batch{BATCH+1}_suggest.csv")
print(f"  Metals: batch{BATCH+1}_metal.csv")

In [ ]:
# Preview the suggested metal configurations
metals_df = pd.read_csv(f'batch{BATCH+1}_metal.csv', header=None)
print(f"\nFirst 5 suggested configurations:")
print(metals_df.head())

# Count metal compositions
for idx in range(min(5, len(metals_df))):
    config = metals_df.iloc[idx].values
    from collections import Counter
    counts = Counter(config[:8])
    print(f"Config {idx}: {dict(counts)}")

## Part 2: DFT Calculations

Now we run DFT calculations for the suggested configurations.

### Running Python Script with GPU/CPU

In [ ]:
# Option A: Run from Python (for interactive use)
from run_dft_batch import DFTBatchRunner

# Initialize runner
runner = DFTBatchRunner(batch_number=BATCH)

# Read suggestions (calculate first 4 configs as example)
runner.read_suggestions(f'batch{BATCH}_metal.csv', start_idx=16, end_idx=20)

# Run O adsorption calculations
# NOTE: This will take hours/days depending on your system!
# results = runner.run_all_calculations(adsorbate='O')

# Save training data
# runner.save_training_data(f'DFT_O_batch{BATCH}.csv')

## Part 3: Process Results and Combine Data

In [ ]:
# After DFT calculations are complete, combine with existing data

# Load existing data
existing_O = np.loadtxt('DFT_O_all.csv', delimiter=',')
existing_OH = np.loadtxt('DFT_OH_all.csv', delimiter=',')

# Load new batch data
new_O = np.loadtxt(f'DFT_O_batch{BATCH}.csv', delimiter=',')
new_OH = np.loadtxt(f'DFT_OH_batch{BATCH}.csv', delimiter=',')

# Combine
combined_O = np.vstack([existing_O, new_O])
combined_OH = np.vstack([existing_OH, new_OH])

# Save updated datasets
np.savetxt('DFT_O_all.csv', combined_O, fmt=['%d']*15+['%.5f'], delimiter=',')
np.savetxt('DFT_OH_all.csv', combined_OH, fmt=['%d']*15+['%.5f'], delimiter=',')

print(f"✓ Combined datasets:")
print(f"  O: {len(existing_O)} + {len(new_O)} = {len(combined_O)} samples")
print(f"  OH: {len(existing_OH)} + {len(new_OH)} = {len(combined_OH)} samples")

## Part 4: Analyze Results

In [ ]:
# Visualize energy distribution
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# O energies
ax1.hist(combined_O[:, -1], bins=30, alpha=0.7, edgecolor='black')
ax1.axvline(new_O[:, -1].mean(), color='r', linestyle='--', label=f'Batch {BATCH} mean')
ax1.set_xlabel('O Adsorption Energy (eV)')
ax1.set_ylabel('Count')
ax1.set_title(f'O Energies (N={len(combined_O)})')
ax1.legend()

# OH energies
ax2.hist(combined_OH[:, -1], bins=30, alpha=0.7, edgecolor='black')
ax2.axvline(new_OH[:, -1].mean(), color='r', linestyle='--', label=f'Batch {BATCH} mean')
ax2.set_xlabel('OH Adsorption Energy (eV)')
ax2.set_ylabel('Count')
ax2.set_title(f'OH Energies (N={len(combined_OH)})')
ax2.legend()

plt.tight_layout()
plt.savefig(f'batch{BATCH}_energy_distribution.png', dpi=300)
plt.show()

In [ ]:
# Calculate descriptor (E_O - E_OH) for new batch
descriptors = new_O[:, -1] - new_OH[:, -1]

print(f"\nBatch {BATCH} Descriptor Statistics:")
print(f"  Mean: {descriptors.mean():.4f} eV")
print(f"  Std: {descriptors.std():.4f} eV")
print(f"  Min: {descriptors.min():.4f} eV")
print(f"  Max: {descriptors.max():.4f} eV")
print(f"  Target (E_opt): 5.30 eV")
print(f"  Closest to target: {descriptors[np.argmin(np.abs(descriptors - 5.3))]:.4f} eV")

## Part 5: Calculate Final Activity

In [ ]:
# Process predictions for activity calculation
counter = ElementCounter()
element_counts = counter.process(
    f'GPR_batch{BATCH}.csv',
    f'batch{BATCH}_count.csv'
)

# Calculate and plot activity
calc = ActivityCalculator(E_opt=5.3, T=300)
activities = calc.calculate(f'batch{BATCH}_count.csv')
calc.plot(f'activity_batch{BATCH}.png')

print(f"\n✓ Activity calculated and plotted")
print(f"\nOptimal composition:")
optimal_idx = np.argmax(activities)
optimal_comp = calc.compositions[optimal_idx]
print(f"  Ni: {optimal_comp[0]:.3f}")
print(f"  Fe: {optimal_comp[1]:.3f}")
print(f"  Co: {optimal_comp[2]:.3f}")
print(f"  Activity: {activities[optimal_idx]:.6f}")

## Summary

Cycle {BATCH} complete! Next steps:

1. ✅ Trained GPR on batches 1-14
2. ✅ Generated suggestions for batch 15
3. ✅ Ran DFT calculations
4. ✅ Combined with existing data
5. ✅ Analyzed results
6. ✅ Calculated activity

**Ready for Cycle {BATCH+1}!**

Simply increment `BATCH = 16` and run again.